# The Mathematics of Random Forest

## 1. Architectural Overview: Bagging and Decision Trees

The Random Forest algorithm is an ensemble learning method that builds upon the foundational concept of Decision Trees (CART). A single decision tree is a "strong learner" but suffers from structural instability: small variations in the training data can lead to vastly different tree topologies. This phenomenon is mathematically defined as high variance.

To mitigate this, Random Forest employs **Bagging** (Bootstrap Aggregating). The algorithm generates B independent datasets by sampling the original data with replacement, and trains a separate decision tree on each. The final prediction is obtained by averaging the outputs (for regression) or by majority vote (for classification).

## 2. The Bias-Variance Tradeoff and Forest Variance

The fundamental mathematical objective of a Random Forest is variance reduction without substantially increasing bias.

According to Leo Breiman's theorem, the variance of the entire forest is not simply the variance of a single tree divided by B. Because the trees are trained on overlapping sets of data, their predictions are correlated.

Let σ² be the variance of a single decision tree, B be the total number of trees (n_estimators), and ρ be the mean pairwise correlation between the trees. The variance of the forest is given by:

Var(Forest) = ρ * σ² + [(1 - ρ) / B] * σ²

As the number of computational resources increases (B → ∞), the second term converges to zero. The asymptotic variance of the model becomes:

lim (B → ∞) Var(Forest) = ρ * σ²

**Critical Analysis:** Adding more trees (increasing B) is computationally expensive and eventually yields diminishing returns. The only mathematical lever available to further reduce the generalization error is to minimize ρ (the correlation between trees).

## 3. Stochastic Perturbation: The Random Subspace Method

To artificially decorrelate the trees and minimize ρ, Random Forest introduces a second layer of stochastic perturbation known as the **Random Subspace Method**, controlled by the hyperparameter `max_features`.

Instead of evaluating the entire feature space P to find the optimal Gini impurity reduction at each node, the algorithm is constrained to evaluate a random subset of features, denoted as m.

The probability of a dominant feature (the strongest biological signal) being selected for evaluation at any given split follows a hypergeometric distribution, conceptually simplified by the expectation:

P(Selecting the dominant feature) = m / P

By restricting m, the algorithm forces different trees to build their internal logic using secondary features. This increases the individual bias of each tree (making them locally weaker) but drastically reduces ρ, yielding a more robust global ensemble.

## 4. Dimensionality Control: √P vs log₂(P)

The selection of the subspace size m is critical. Scikit-Learn offers two primary mathematical functions to determine this limit: the square root and the base-2 logarithm.

* **Square Root (sqrt):** m = √P
* **Logarithm (log2):** m = log₂(P)

The computational and structural divergence between these two limits becomes significant only in high-dimensional spaces. The logarithmic function exhibits much slower asymptotic growth compared to the square root:

∀ P > 16, log₂(P) < √P

In high-dimensional datasets (e.g., P = 10000), using `log2` yields m ≈ 13, whereas `sqrt` yields m = 100. The `log2` parameter enforces extreme decorrelation, accelerating matrix operations but demanding deeper trees to capture complex relationships.

### The Low-Dimensional Anomaly

In restricted dimensional spaces, these two functions converge to virtually identical integers. For instance, evaluating an optimized vector space of P = 12 features:

* m_sqrt = √12 ≈ 3.46 → 3 features
* m_log2 = log₂(12) ≈ 3.58 → 3 features

Evaluating both `sqrt` and `log2` inside a GridSearchCV on a low-dimensional matrix results in redundant computational overhead, forcing the processor to evaluate duplicate subsets without altering the underlying probability P.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

X_train = pd.read_csv("data/X_train_purged_df.csv", sep=',')
mapping = {"Absence" : 0, "Presence" : 1}
X_train["Heart Disease"] = X_train["Heart Disease"].map(mapping)
y = X_train["Heart Disease"]
X_train.drop(columns="Heart Disease", inplace=True)

X_test = pd.read_csv("data/X_test_purged_df.csv", sep=',')
df_test = pd.read_csv("data/test.csv", sep=",")

print(len(X_train))

20000


In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 500],           # Number of trees in the forest
    'max_depth': [None, 5, 8, 12],             # Maximum depth (Control Overfitting)
    'min_samples_split': [2, 5, 10],           # Min samples required to split a node
    'min_samples_leaf': [1, 2, 4],             # Min samples required in a leaf
    'max_features': ['sqrt', 'log2']           # Number of features to consider at each split
}

rf_base = RandomForestClassifier()

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring="accuracy",
    verbose=True,
    refit=True)

grid_search.fit(X_train, y)

print("\n--- OPTIMIZATION RESULTS ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy Score: {grid_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

--- OPTIMIZATION RESULTS ---
Best Hyperparameters: {'max_depth': 12, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 500}
Best CV Accuracy Score: 87.69%


In [10]:
best_model = RandomForestClassifier(**grid_search.best_params_)
best_model.fit(X_train, y)

X_test_no_id = X_test.copy().drop(columns="id")

y_pred = best_model.predict(X_test_no_id)
y_pred = np.where(y_pred == "Presence", 1, 0)
y_pred_df = pd.DataFrame({
    "id" : df_test.id,
    "Heart Disease" : y_pred
})

y_pred_df.to_csv("data/kaggle_sub_rf_max_depth12_max_featuressqrt_minsamplesleaf4_minsamplessplit10_nestimators200.csv", index=False)
y_pred_df.head()

,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0
